# ServerBench on Colab (L4): Claude vs Codex

這本 notebook 在 **一張 Colab L4** 上跑 ServerBench 的 serving-system evolve，用來比較 Claude Code 和 Codex。

Colab 不能跑 Docker，所以這裡用的是 repo 裡新增的 **local backend**：
- agent 以一般使用者 `agent` 身分，跑在只有 loopback 的 network namespace 裡。唯一的對外連線是 egress broker，只放行模型供應商的 API，每次連線都會記錄。
- 私有資料（答案池、驗證結果、其他 run、已發佈的 lineage）都設成 root-only。agent 結束、所有程序確認清除之後，才在新的沙箱裡跑最終驗證。
- 版本發佈、`./full-test`、journal、trajectory、計分，用的都是和 Docker 版相同的程式。

**順序**（每一步的結果都會存到 Google Drive，斷線後可以接著做）：
1. 設定 runtime（每個新 session 都要重跑，約 30–60 分鐘）
2. **校準**：在 L4 上量 SGLang 的參考點和 p90 bar。每個任務只需要做一次，約 5–6 小時 GPU，可以分好幾個 session 完成
3. 跑 Claude（3 小時 + 約 45 分鐘驗證）
4. 跑 Codex（3 小時 + 約 45 分鐘驗證）
5. 比較

> 執行期間請保持這個分頁開著。Colab Pro 沒有背景執行，分頁關掉之後 runtime 會被回收。

## 0. 參數

In [ ]:
TASK = "qwen3_5_2b_standard_1xL4"
HOURS = 3                       # each agent's wall-clock budget
CLAUDE_MODEL, CLAUDE_EFFORT = "claude-opus-5", "xhigh"
CODEX_MODEL, CODEX_EFFORT = "gpt-5.6-sol", "max"
REPO_URL = "https://github.com/allen3325/Server-Bench-Colab.git"
BRANCH = "colab"
REPO = "/content/Server-Bench-Colab"
DRIVE_DIR = "/content/drive/MyDrive/serverbench-colab"
CALIBRATION = "/content/sb-calibration"
import os
os.environ.update(TASK=TASK, CALIBRATION=CALIBRATION, HF_HOME="/opt/serverbench/hf")

## 1. 確認 GPU，取得 repo

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import os, subprocess
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
%cd {REPO}

## 2. 從 Drive 還原先前的結果

會掛載 Drive，把先前存的校準結果、已校準的任務檔和 run 的結果還原回來。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!bash serverbench/scripts/colab/persist.sh restore {DRIVE_DIR}

## 3. 憑證

請在 Colab 左側的 🔑 **Secrets** 設定以下其中一組（並打開 notebook access）：

| Claude | 說明 |
|---|---|
| `CLAUDE_CODE_OAUTH_TOKEN` | 用訂閱方案：在自己電腦上執行 `claude setup-token` 取得 |
| `ANTHROPIC_API_KEY` | 用 API 金鑰計費 |

| Codex | 說明 |
|---|---|
| `OPENAI_API_KEY` | 會用 `codex login --with-api-key` 寫成 `~/.codex/auth.json` |

金鑰只會寫進 root 才能讀的 `serverbench/artifacts/secrets/`，再由 harness 交給該次 run 的沙箱。

In [ ]:
from google.colab import userdata
import pathlib, subprocess, os

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

secrets_dir = pathlib.Path(REPO, "serverbench/artifacts/secrets")
secrets_dir.mkdir(parents=True, exist_ok=True)
os.chmod(secrets_dir, 0o700)
claude_env = secrets_dir / "claude.env"
if secret("CLAUDE_CODE_OAUTH_TOKEN"):
    claude_env.write_text(f"CLAUDE_CODE_OAUTH_TOKEN={secret('CLAUDE_CODE_OAUTH_TOKEN')}\n")
    print("Claude: subscription OAuth token")
elif secret("ANTHROPIC_API_KEY"):
    claude_env.write_text("ANTHROPIC_BASE_URL=https://api.anthropic.com\n"
                          f"ANTHROPIC_API_KEY={secret('ANTHROPIC_API_KEY')}\n")
    print("Claude: API key")
else:
    print("Claude: no secret set")
os.chmod(claude_env, 0o600) if claude_env.exists() else None

## 4. 設定 runtime（每個 session 都要跑）

In [ ]:
!bash serverbench/scripts/colab/setup.sh 2>&1 | tail -60

In [ ]:
# Codex login (after setup installed the CLI)
key = secret("OPENAI_API_KEY")
if key:
    subprocess.run(["/opt/serverbench/bin/codex", "login", "--with-api-key"],
                   input=key, text=True, check=True)
    print("Codex: logged in with the API key ->", pathlib.Path.home() / ".codex/auth.json")
else:
    print("Codex: no OPENAI_API_KEY secret")

## 5. 校準：在 L4 上量 SGLang 的參考點和 p90 bar（每個任務只做一次）

方法和 repo 原本的一樣（`docs/harness/bar-derivation.md`）：
1. **search**：offered concurrency 1, 2, 4, 8, 16 各跑一個 1,800 秒窗口，取「gross $/$ ÷ p90 端到端延遲」最高的點
2. **seeds**：在選定的點再跑 5 個不同 seed，bar = 5 次中最差的 p90
3. **record**：寫回任務的 `reference.yaml` 和 `contract.yaml`

每個點量完都可以續跑：已經量過的點會直接讀回來。每跑完一步就存到 Drive。

In [ ]:
MODEL_DIR = subprocess.run(["bash", "-c",
    "ls -d /opt/serverbench/hf/hub/models--Qwen--Qwen3.5-2B/snapshots/*/ | head -1"],
    capture_output=True, text=True).stdout.strip()
print(MODEL_DIR)
!serverbench/.venv/bin/python serverbench/scripts/colab/calibrate.py search --task {TASK} --model-dir {MODEL_DIR} --out-dir {CALIBRATION}
!bash serverbench/scripts/colab/persist.sh save {DRIVE_DIR}

In [ ]:
!serverbench/.venv/bin/python serverbench/scripts/colab/calibrate.py seeds --task {TASK} --model-dir {MODEL_DIR} --out-dir {CALIBRATION}
!bash serverbench/scripts/colab/persist.sh save {DRIVE_DIR}

In [ ]:
!serverbench/.venv/bin/python serverbench/scripts/colab/calibrate.py record --task {TASK} --out-dir {CALIBRATION}
!bash serverbench/scripts/colab/persist.sh save {DRIVE_DIR}
!grep -A3 service_levels serving_tasks/{TASK}/contract.yaml

## 6. 跑 agent

開始前會**先卸載 Drive**（agent 不可以看到你的 Drive），跑完再重新掛載、存結果。

run 在背景執行（`setsid nohup`），下面的監看 cell 每 5 分鐘印一次進度，也能讓 session 保持活動。時間到之後 driver 會凍結最後版本，再跑最終驗證（約 45 分鐘）。

In [ ]:
import subprocess, time, pathlib, json, os

def start_run(cli, model, effort):
    from google.colab import drive
    if os.path.ismount("/content/drive"):
        drive.flush_and_unmount()
    env = dict(os.environ, AGENT_CLI=cli, AGENT_MODEL=model, AGENT_EFFORT=effort,
               PATH="/opt/serverbench/bin:" + os.environ["PATH"])
    if cli == "claude" and (pathlib.Path(REPO) / "serverbench/artifacts/secrets/claude.env").exists():
        env["AGENT_ENV"] = str(pathlib.Path(REPO) / "serverbench/artifacts/secrets/claude.env")
    log = pathlib.Path(f"/content/run-{cli}.log")
    proc = subprocess.Popen(
        ["setsid", "nohup", "bash", "serverbench/scripts/server_evolve/local-evolve.sh",
         TASK, "--hours", str(HOURS)],
        cwd=REPO, env=env, stdout=log.open("w"), stderr=subprocess.STDOUT,
        stdin=subprocess.DEVNULL)
    print(f"started {cli} ({model}); log {log}")
    return proc, log

def latest_run():
    runs = sorted(pathlib.Path(REPO, "serverbench/evolve_space/evolve", TASK.lower()).glob("*/journal.jsonl"),
                  key=lambda p: p.stat().st_mtime)
    return runs[-1].parent if runs else None

def watch(proc, log, every=300):
    while proc.poll() is None:
        run = latest_run()
        events = []
        if run is not None:
            for line in (run / "journal.jsonl").read_text().splitlines()[-400:]:
                try:
                    events.append(json.loads(line).get("event"))
                except ValueError:
                    pass
        versions = events.count("version_registered")
        print(time.strftime("%H:%M:%S"), "| last event:", events[-1] if events else "-",
              "| versions:", versions, "| run:", run.name if run else "-", flush=True)
        time.sleep(every)
    print("finished with exit code", proc.returncode)
    print(log.read_text()[-3000:])

def save_results():
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run(["bash", "serverbench/scripts/colab/persist.sh", "save", DRIVE_DIR], cwd=REPO)

### 6a. Claude

In [ ]:
proc, log = start_run("claude", CLAUDE_MODEL, CLAUDE_EFFORT)
watch(proc, log)
save_results()

### 6b. Codex

In [ ]:
proc, log = start_run("codex", CODEX_MODEL, CODEX_EFFORT)
watch(proc, log)
save_results()

## 7. 比較

讀兩個 lineage 的最終驗證結果（`FINAL.json`）與 `evaluation_progress/<task>.csv`。

In [ ]:
import glob, json, pathlib
rows = []
for final in sorted(glob.glob(f"{REPO}/server_output/*/FINAL.json")):
    body = json.loads(pathlib.Path(final).read_text())
    if TASK not in json.dumps(body):
        continue
    rows.append((pathlib.Path(final).parent.name, body))
for lineage, body in rows:
    print("==", lineage)
    print(json.dumps({k: body.get(k) for k in ("status", "head", "verdict", "dollars_per_dollar",
                                                "gross_dollars_per_dollar", "because") if k in body},
                     indent=1)[:1500])
csv = pathlib.Path(REPO, "evaluation_progress", f"{TASK}.csv")
print(csv.read_text() if csv.exists() else "(no ledger rows yet)")